In [1]:
import sys
import os
import pandas as pd
from threading import Thread
from multiprocessing import Process, Manager
import datetime
import uuid
import json
import _pickle as pickle
import time
import traceback
from psycopg2 import extras
from binance import AsyncClient, BinanceSocketManager # For MarginCallback
import asyncio # For MarginCallback
import numpy as np
import jwt
import websocket
import pandas as pd

upper_dir = os.path.dirname("/home/trade_core/")
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.binance_plug import InitBinanceAdaptor
from exchange_plugin.upbit_plug import InitUpbitAdaptor
from etc.register_monitor_msg import RegisterMonitorMsg

In [2]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 1
# node = config['node']
node = 'trade_core1'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets = config['node_settings'][node]['enabled_markets']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [3]:
class InitTrigger:
    def __init__(self, admin_id, node, server_check_status_list, get_premium_df, enabled_markets, register_monitor_msg, remote_redis_client, mongo_client, postgres_client, logging_dir):
        self.admin_id = admin_id
        self.node = node
        self.server_check_status_list = server_check_status_list
        self.get_premium_df = get_premium_df
        self.enabled_markets = enabled_markets
        self.register_monitor_msg = register_monitor_msg
        self.remote_redis_client = remote_redis_client
        self.db_dict = db_dict
        self.mongo_client = mongo_client
        self.postgres_client = postgres_client
        self.logging_dir = logging_dir
        self.logger = KimpBotLogger(logging_dir, "trigger").logger
        self.user_info_df = None
        self.exchange_config_df = None
        self.trade_df = None
        self.load_user_info_thread = Thread(target=self.load_user_info, daemon=True)
        self.load_exchange_config_thread = Thread(target=self.load_exchange_config, daemon=True)
        self.load_user_info_thread.start()
        self.load_exchange_config_thread.start()
        while self.user_info_df is not None and self.exchange_config_df is not None:
            time.sleep(0.2)
        self.logger.info("user_info_df and exchange_config_df have been initialized")

    def is_table_empty(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cur.fetchone()[0]
            return count == 0

    def get_column_names(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_schema = 'public' AND table_name = %s
                ORDER BY ordinal_position;
            """, (table_name,))
            return [row[0] for row in cur.fetchall()]

    def load_user_info(self, table_name='user_info', loop_interval_secs=1):
        while True:
            try:
                # First check whether the table is empty
                conn = self.postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                if self.is_table_empty(conn, table_name):
                    # Get the column names
                    column_names = self.get_column_names(conn, table_name)
                    # Create empty dataframe
                    self.user_info_df = pd.DataFrame(columns=column_names)
                else:
                    curr.execute(f"SELECT * FROM {table_name}")
                    self.user_info_df = pd.DataFrame(curr.fetchall())
                self.postgres_client.pool.putconn(conn)
                time.sleep(loop_interval_secs)
            except Exception as e:
                # rollback the transaction if any error while inserting
                self.postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in load_user_info: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"load_user_info", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)

    def load_exchange_config(self, table_name='exchange_config', loop_interval_secs=1):
        while True:
            try:
                # First check whether the table is empty
                conn = self.postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                if self.is_table_empty(conn, table_name):
                    # Get the column names
                    column_names = self.get_column_names(conn, table_name)
                    # Create empty dataframe
                    self.exchange_config_df = pd.DataFrame(columns=column_names)
                else:
                    curr.execute(f"SELECT * FROM {table_name}")
                    self.exchange_config_df = pd.DataFrame(curr.fetchall())
                self.postgres_client.pool.putconn(conn)
                time.sleep(loop_interval_secs)
            except Exception as e:
                # rollback the transaction if any error while inserting
                self.postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in load_exchange_config: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"load_exchange_config", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)
        
    def start_trigger_loop(self, market_combi_code, table_name='trade', loop_interval_secs=1):
        target_market_code, origin_market_code = market_combi_code.split(':')
        while True:
            try:
                # First check whether the table is empty
                conn = self.postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                if self.is_table_empty(conn, table_name):
                    # Get the column names
                    column_names = self.get_column_names(conn, table_name)
                    # Create empty dataframe
                    self.trade_df = pd.DataFrame(columns=column_names)
                else:
                    curr.execute(f"SELECT * FROM {table_name} WHERE target_market_code = %s AND origin_market_code = %s", (target_market_code, origin_market_code))
                    self.trade_df = pd.DataFrame(curr.fetchall())
                self.postgres_client.pool.putconn(conn)
                time.sleep(loop_interval_secs)
            except Exception as e:
                # rollback the transaction if any error while inserting
                self.postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in start_trigger_loop: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"start_trigger_loop", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)


In [4]:
trigger = InitTrigger(admin_id, node, None, enabled_markets, register_monitor_msg, None, db_dict, None)

TypeError: __init__() missing 2 required positional arguments: 'postgres_client' and 'logging_dir'

In [6]:
trigger.exchange_config_df

,id,user_uuid,registered_datetime,service_datetime_end,target_market_code,origin_market_code,target_market_uid,origin_market_uid,target_market_referral_use,origin_market_referral_use,...,origin_market_margin_call,target_market_safe_reverse,origin_market_safe_reverse,target_market_risk_threshold_p,origin_market_risk_threshold_p,repeat_limit_p,repeat_limit_direction,repeat_num_limit,on_off,remark
0,1,test_uuid,2023-12-18 15:54:54.893783,2023-12-25 15:54:54.893692,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,None,None,False,False,...,2,True,True,None,None,4.0,UPWARD,None,True,None
1,2,test_uuid,2023-12-18 15:54:58.902161,2023-12-25 15:54:58.902085,UPBIT_SPOT/KRW,OKX_USD_M/USDT,None,None,False,False,...,2,True,True,None,None,4.0,UPWARD,None,True,None


In [5]:
def is_table_empty(conn, table_name):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cur.fetchone()[0]
        return count == 0
    
def get_column_names(conn, table_name):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position;
        """, (table_name,))
        return [row[0] for row in cur.fetchall()]

In [4]:
test_client = InitDBClient(**{**db_dict, 'database': 'trade_core'})
test_client.create_all_table()

InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|TABLE: user_info already exists.
InitDBClient|TABLE: exchange_config already exists.


In [16]:

start = time.time()
for i in range(1000):
    conn = test_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    curr.execute('SELECT * FROM user_info')
    df = pd.DataFrame(curr.fetchall())
    # print(df)
    test_client.pool.putconn(conn)
print(time.time()-start)

0.6362197399139404


In [11]:
conn = test_client.get_conn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
SELECT *
FROM user_info
INNER JOIN exchange_config ON user_info.user_uuid = exchange_config.user_uuid;
"""
curr.execute(sql)
result = curr.fetchall()
pd.DataFrame(result)


,id,user_uuid,email,telegram_id,telegram_name,registered_datetime,status,alarm_num,alarm_period,remark,...,target_market_margin_call,origin_market_margin_call,target_market_safe_reverse,origin_market_safe_reverse,target_market_risk_threshold_p,origin_market_risk_threshold_p,repeat_limit_p,repeat_limit_direction,repeat_num_limit,on_off
0,1,test_uuid,ckddjs116@gmail.com,1390695186,None,2023-12-18 15:54:54.893783,None,1,1,None,...,None,2,True,True,None,None,4.0,UPWARD,None,True
1,2,test_uuid,ckddjs116@gmail.com,1390695186,None,2023-12-18 15:54:58.902161,None,1,1,None,...,None,2,True,True,None,None,4.0,UPWARD,None,True


In [11]:
# For Adding user_info

sql = """
INSERT INTO user_info (user_uuid, email, telegram_id, telegram_name, registered_datetime, status, alarm_num, alarm_period, remark) 
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
val = ['test_uuid', 'ckddjs116@gmail.com', 1390695186, None, datetime.datetime.utcnow(), None, 1, 1, None]
curr.execute(sql, val)
conn.commit()

In [9]:
# For adding exchange_config

sql = """
INSERT INTO exchange_config (user_uuid, registered_datetime, service_datetime_end, target_market_code, origin_market_code, target_market_uid, origin_market_uid,
target_market_referral_use, origin_market_referral_use, target_market_cross, target_market_leverage, origin_market_cross, origin_market_leverage, target_market_margin_call, origin_market_margin_call,
target_market_safe_reverse, origin_market_safe_reverse, target_market_risk_threshold_p, origin_market_risk_threshold_p, repeat_limit_p, repeat_limit_direction, repeat_num_limit,
on_off, remark)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

datetime_now = datetime.datetime.utcnow()
service_datetime_end = datetime_now + datetime.timedelta(days=7)
val = ['test_uuid', datetime.datetime.utcnow(), service_datetime_end, 'UPBIT_SPOT/KRW', 'OKX_USD_M/USDT', None, None,
       False, False, None, None, True, 4, None, 2, True, True, None, None, 4, 'UPWARD', None, True, None]
curr.execute(sql, val)
conn.commit()



In [32]:
conn.rollback()